In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler


df = pd.read_csv('bank.csv')
target_col = 'deposit'
df[target_col] = df[target_col].map({'yes': 1, 'no': 0})
X = pd.get_dummies(df.drop(target_col, axis=1), drop_first=True)
y = df[target_col]
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=2137)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]


print("Wyniki dla różnych progów decyzyjnych:")
for threshold in [0.1, 0.25, 0.5, 0.75, 0.9]:
    y_pred_t = (y_pred >= threshold)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    print(f"Próg {threshold:^4}: TruePos={tp:<4} FalsePos={fp:<4} TrueNeg={tn:<4} FalseNeg={fn:<4}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
tprs = []
fprs = []
min_cost = float('inf')
best_threshold = 0
best_matrix = (0, 0, 0, 0)
for t in thresholds:
    y_pred_t = (y_pred >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    current_cost = (10*fp) + (fn*3)
    if current_cost < min_cost:
        min_cost = current_cost
        best_threshold = t
        best_matrix = (tn, fp, fn, tp)
        
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    tprs.append(tpr)
    fprs.append(fpr)
    
plt.figure(figsize=(8, 6))
plt.plot(fprs, tprs, color='blue', lw=2, label='Krzywa ROC')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Klasyfikator losowy')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Krzywa ROC dla modelu regresji logistycznej')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()
    

In [ ]:
print(f"Optymalny próg decyzyjny: {best_threshold:.2f}")
print(f"Minimalny koszt całkowity: {min_cost}")
print(f"Macierz błędów dla optymalnego progu: TN={best_matrix[0]}, FP={best_matrix[1]}, FN={best_matrix[2]}, TP={best_matrix[3]}")

In [ ]:
wagi = np.abs(model.coef_[0])
top_2_features = X.columns[np.argsort(wagi)[-2:]].tolist()

X_2_train, X_2_test, y_train_2, y_test_2 = train_test_split(X[top_2_features], y, test_size=0.2, random_state=2137)

scaler_2 = StandardScaler()
X_2_train_scaled = scaler_2.fit_transform(X_2_train)
X_2_test_scaled = scaler_2.transform(X_2_test)

model_2 = LogisticRegression().fit(X_2_train_scaled, y_train_2)

x_min, x_max = X_2_test_scaled[:, 0].min() - 1, X_2_test_scaled[:, 0].max() + 1
y_min, y_max = X_2_test_scaled[:, 1].min() - 1, X_2_test_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

Z = model_2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, levels=[0, best_threshold, 1], alpha=0.3, colors=['red', 'blue'])
plt.contour(xx, yy, Z, levels=[best_threshold], linewidths=2, colors='black')
plt.scatter(X_2_test_scaled[:, 0], X_2_test_scaled[:, 1], c=y_test_2, edgecolors='k', alpha=0.7, cmap=plt.cm.RdYlBu)
plt.title(f"Granica decyzyjna (próg {best_threshold:.2f}) dla: {top_2_features[0]}, {top_2_features[1]}")
plt.show()